In [4]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.data_prep import load_and_preprocess_data
from src.resampling import apply_smote

# 1. Load clean split
X_train, X_test, y_train, y_test = load_and_preprocess_data('../data/raw/creditcard.csv')

print(f"Original y_train distribution:\n{y_train.value_counts()}\n")

# 2. Apply default SMOTE (1:1 ratio)
X_res, y_res = apply_smote(X_train, y_train, sampling_strategy=0.1)

print(f"Resampled y_train distribution (sampling_strategy=1.0):\n{y_res.value_counts()}")

Original y_train distribution:
Class
0    227451
1       394
Name: count, dtype: int64

Resampled y_train distribution (sampling_strategy=1.0):
Class
0    227451
1     22745
Name: count, dtype: int64


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, average_precision_score, roc_auc_score

# 1. Train Baseline Model (No Resampling)
rf_baseline = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_baseline.fit(X_train, y_train)

# Predict probabilities on untouched X_test
y_pred_base = rf_baseline.predict(X_test)
y_proba_base = rf_baseline.predict_proba(X_test)[:, 1]

print("=== BASELINE MODEL (No Resampling) ===")
print(classification_report(y_test, y_pred_base, digits=4))
print(f"PR-AUC Score: {average_precision_score(y_test, y_proba_base):.4f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba_base):.4f}\n")


# 2. Train Model on SMOTE Data (10% Ratio)
rf_smote = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_smote.fit(X_res, y_res)

# Predict probabilities on untouched X_test
y_pred_smote = rf_smote.predict(X_test)
y_proba_smote = rf_smote.predict_proba(X_test)[:, 1]

print("=== SMOTE MODEL (10% Resampled) ===")
print(classification_report(y_test, y_pred_smote, digits=4))
print(f"PR-AUC Score: {average_precision_score(y_test, y_proba_smote):.4f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba_smote):.4f}")

=== BASELINE MODEL (No Resampling) ===
              precision    recall  f1-score   support

           0     0.9997    0.9999    0.9998     56864
           1     0.9412    0.8163    0.8743        98

    accuracy                         0.9996     56962
   macro avg     0.9704    0.9081    0.9371     56962
weighted avg     0.9996    0.9996    0.9996     56962

PR-AUC Score: 0.8734
ROC-AUC Score: 0.9630

=== SMOTE MODEL (10% Resampled) ===
              precision    recall  f1-score   support

           0     0.9997    0.9998    0.9998     56864
           1     0.8925    0.8469    0.8691        98

    accuracy                         0.9996     56962
   macro avg     0.9461    0.9234    0.9344     56962
weighted avg     0.9996    0.9996    0.9996     56962

PR-AUC Score: 0.8787
ROC-AUC Score: 0.9697


In [6]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score, 
    recall_score, 
    f1_score, 
    average_precision_score, 
    roc_auc_score
)
from src.resampling import apply_smote

# Include None for the unadjusted baseline
strategies = [None, 0.05, 0.1, 0.5, 1.0]
results = []

for strat in strategies:
    if strat is None:
        print("Evaluating Baseline (No SMOTE)...")
        X_train_curr, y_train_curr = X_train, y_train
        strat_label = "Baseline (0.17%)"
    else:
        print(f"Evaluating SMOTE ratio = {strat}...")
        X_train_curr, y_train_curr = apply_smote(X_train, y_train, sampling_strategy=strat)
        strat_label = f"SMOTE ({strat})"
        
    # Fit Random Forest
    model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_train_curr, y_train_curr)
    
    # Predict on untouched X_test
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    # Store Fraud Class (1) Metrics
    results.append({
        "Strategy": strat_label,
        "Train Fraud Ratio": f"{y_train_curr.mean():.2%}",
        "Precision": round(precision_score(y_test, y_pred, pos_label=1), 4),
        "Recall": round(recall_score(y_test, y_pred, pos_label=1), 4),
        "F1-Score": round(f1_score(y_test, y_pred, pos_label=1), 4),
        "PR-AUC": round(average_precision_score(y_test, y_proba), 4),
        "ROC-AUC": round(roc_auc_score(y_test, y_proba), 4)
    })

# Format results table
results_df = pd.DataFrame(results)
results_df

Evaluating Baseline (No SMOTE)...
Evaluating SMOTE ratio = 0.05...
Evaluating SMOTE ratio = 0.1...
Evaluating SMOTE ratio = 0.5...
Evaluating SMOTE ratio = 1.0...


,Strategy,Train Fraud Ratio,Precision,Recall,F1-Score,PR-AUC,ROC-AUC
0,Baseline (0.17%),0.17%,0.9412,0.8163,0.8743,0.8734,0.9630
1,SMOTE (0.05),4.76%,0.9140,0.8673,0.8901,0.8802,0.9648
2,SMOTE (0.1),9.09%,0.8925,0.8469,0.8691,0.8787,0.9697
3,SMOTE (0.5),33.33%,0.8646,0.8469,0.8557,0.8769,0.9678
4,SMOTE (1.0),50.00%,0.8632,0.8367,0.8497,0.8706,0.9734


In [7]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score
from src.resampling import apply_smote
from sklearn.ensemble import RandomForestClassifier

# 1. Train model on optimal SMOTE ratio (0.05)
X_res_opt, y_res_opt = apply_smote(X_train, y_train, sampling_strategy=0.05)
best_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
best_model.fit(X_res_opt, y_res_opt)

# 2. Get predicted probabilities on untouched X_test
y_proba = best_model.predict_proba(X_test)[:, 1]

# 3. Sweep probability thresholds from 0.10 to 0.50
thresholds = [0.10, 0.20, 0.30, 0.40, 0.50]
thresh_results = []

for t in thresholds:
    # Custom prediction rule based on threshold
    y_pred_t = (y_proba >= t).astype(int)
    
    thresh_results.append({
        "Threshold": t,
        "Precision": round(precision_score(y_test, y_pred_t), 4),
        "Recall": round(recall_score(y_test, y_pred_t), 4),
        "F1-Score": round(f1_score(y_test, y_pred_t), 4),
        "Fraud Caught": f"{int(recall_score(y_test, y_pred_t) * 98)} / 98"
    })

pd.DataFrame(thresh_results)

,Threshold,Precision,Recall,F1-Score,Fraud Caught
0,0.1,0.3248,0.9082,0.4785,89 / 98
1,0.2,0.6331,0.8980,0.7426,88 / 98
2,0.3,0.7373,0.8878,0.8056,87 / 98
3,0.4,0.8431,0.8776,0.8600,86 / 98
4,0.5,0.9140,0.8673,0.8901,85 / 98
